# DX 704 Week 9 Project | Enron e-mail SPAM classifier

- This week's project will build an email spam classifier based on the Enron email data set.
- You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
- Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Imports

In [124]:
import numpy as np
import pandas as pd
import re
import json
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, confusion_matrix

## Functions

In [125]:
def show_dict_slice(dict, divisor):
    portion_len = len(dict) // divisor
    output = dict(list(dict.items())[:portion_len])
    print(output)

### process_and_extract_features

In [126]:

def process_and_extract_features(text: str) -> dict:
        """
        Cleans the input text and extracts word counts as features.
        
        The cleaning steps include:
        1. Converting text to lowercase.
        2. Removing punctuation (non-alphanumeric characters except spaces/newlines).
        3. Splitting the text into tokens (words).
        4. Removing common English stopwords (optional, but highly recommended for NB).
        
        Args:
            text (str): The combined subject and message text.
            
        Returns:
            dict: A dictionary where keys are words (features) and values are counts.
        """
        
        #  list of English stopwords (to be expanded)
        stopwords = set([
            'a', 'an', 'the', 'and', 'but', 'or', 'of', 'to', 'in', 'on', 
            'is', 'it', 'for', 'with', 'at', 'by', 'as', 'do', 'he', 'she', 
            'i', 'me', 'my', 'you', 'your', 'we', 'our', 'they', 'their', 
            'this', 'that', 'was', 'were', 'be', 'are', 'not', 'have', 'had',
            'from', 'so', 'if', 'all', 'out', 'up', 'down', 'about', 'just'
        ])

        text    = str(text).lower()
        
        # Remove anything that is not a word character or space (i.e., punctuation)
        # Using  RegEx to replace non-word/non-space characters with a space
        text    = re.sub(r'[^\w\s]', ' ', text)
        
        # Tokenize (split by whitespace) and filter out short tokens and stopwords
        tokens  = []
        
        for word in text.split():
            if len(word) > 1 and word not in stopwords:
                tokens.append(word)


        # tokens  = [word for word in text.split() if len(word) > 1 and word not in stopwords]
        
        # Count token repetitions
        feature_counts = Counter(tokens)
        print(f"Extracting {len(feature_counts)} features from text of length {len(text)}")
        
        feature_counts_dict = dict(feature_counts)
        # print(f"Features: {list(feature_counts_dict.items())}\n")

        return feature_counts_dict

### train_naive_bayes_model 

Calculates the smoothed likelihoods:$$\text{Likelihood: } P(f|C) = \frac{\text{Count}(f, C) + \alpha}{\sum_{f'} \text{Count}(f', C) + \alpha \cdot |\text{Vocabulary}|}$$These resulting values, $P(f|C)$, are small positive floating-point numbers (between 0 and 1).

In [127]:
def train_naive_bayes_model(train_df: pd.DataFrame, 
                            feature_col: str    = 'features_dict', 
                            label_col: str      = 'Spam/Ham', 
                            alpha: int          = 1) -> pd.DataFrame:

    """
    Trains a Multinomial Naive Bayes model with Laplace smoothing (alpha=1)
    and returns a DataFrame of feature likelihoods P(f|C).
    
    Arguments:
        train_df: Training DataFrame containing the specified feature and label columns.
        feature_col: The name of the column containing the word count dictionary.
        label_col: The name of the column containing the numeric class label (0=ham, 1=spam).
        alpha: Smoothing parameter.
        
    Returns:
        DataFrame with columns: 'feature', 'ham_probability', 'spam_probability'.
    """
    print("Training Naive Bayes Model and generating df of feature probabilities...")

    # ---  Aggregate Feature Counts ---
    feature_counts_ham  = Counter()
    feature_counts_spam = Counter()
    total_counts_ham    = 0
    total_counts_spam   = 0

    for _, row in train_df.iterrows():
        label       = row[label_col]
        features    = row[feature_col]
        
        for feature, count in features.items():
            if label == 0: # Ham
                feature_counts_ham[feature] += count
                total_counts_ham            += count
            else: # Spam
                feature_counts_spam[feature] += count
                total_counts_spam            += count

    # ---  Determine Vocabulary ---
    vocabulary = set(feature_counts_ham.keys()) | set(feature_counts_spam.keys())
    vocab_size = len(vocabulary)

    # ---  Calculate Likelihoods (P(f|C) with Laplace Smoothing) ---
    likelihood_data = []
    
    # Denominators: (Total word count in class + alpha * |Vocabulary|)
    denominator_ham  = total_counts_ham + alpha * vocab_size
    denominator_spam = total_counts_spam + alpha * vocab_size

    for feature in vocabulary:
        count_ham  = feature_counts_ham.get(feature, 0)
        count_spam = feature_counts_spam.get(feature, 0)
        
        # Calculate P(f|C)
        prob_ham  = (count_ham + alpha) / denominator_ham
        prob_spam = (count_spam + alpha) / denominator_spam
        
        likelihood_data.append({
            'feature':          feature,
            'ham_probability':  prob_ham,
            'spam_probability': prob_spam
        })

    return pd.DataFrame(likelihood_data)



### classify_document

In [128]:
def classify_document(document_features: dict, model: dict) -> tuple[float, float]:

    """
    Computes the normalized posterior probabilities that a document belongs to the 'Ham' or 'Spam' class
    using a Naive Bayes classifier with log-probabilities for numerical stability.

    Parameters:
        ----------
        document_features : dict
            A dictionary representing the features of the document, where keys are feature names (e.g., words or tokens)
            and values are their corresponding counts or frequencies in the document.

        model : dict
            A dictionary containing the trained Naive Bayes model parameters:
            - 'prior': A tuple or list with prior probabilities for the Ham and Spam classes, respectively.
            - 'likelihood': A dictionary mapping each feature to a tuple of likelihoods (P(feature|Ham), P(feature|Spam)).

    Returns:
        -------
        tuple[float, float]
            A tuple containing the normalized posterior probabilities:
            - P_Ham: Probability that the document is Ham.
            - P_Spam: Probability that the document is Spam.

        Notes:
        ------
        - The function uses log-space arithmetic to prevent numerical underflow when multiplying many small probabilities.
        - It applies the log-sum-exp trick to safely normalize the log-probabilities.
        - If the computed denominator is zero (which may happen due to extreme underflow), it returns equal probabilities (0.5, 0.5).
    """

    # print(f"Calculating predictions for: {document_features}")

    # Initialize log-priors
    log_numerator_ham  = np.log(model['prior'][0])
    log_numerator_spam = np.log(model['prior'][1])
    
    # Add weighted log-likelihoods: sum(count(f) * log(P(f|C)))
    for feature, count in document_features.items():
        if feature in model['likelihood']:
            log_prob_ham  = np.log(model['likelihood'][feature][0])
            log_prob_spam = np.log(model['likelihood'][feature][1])
            
            log_numerator_ham  += count * log_prob_ham
            log_numerator_spam += count * log_prob_spam

    # Log-sum-exp trick for safe normalization
    max_log = max(log_numerator_ham, log_numerator_spam)
    
    # Convert back to unnormalized probabilities
    num_ham  = np.exp(log_numerator_ham - max_log)
    num_spam = np.exp(log_numerator_spam - max_log)

    # Normalize
    denominator = num_ham + num_spam
    
    if denominator == 0:
        return (0.5, 0.5)
    
    P_Ham = num_ham / denominator
    P_Spam = num_spam / denominator
    
    return (P_Ham, P_Spam)

### calculate_roc_curve

In [129]:

# Assumes dataframe's true label = 'Spam/Ham' and that the predicted label = 'spam'  

def calculate_roc_curve(df: pd.DataFrame, threshold: float) -> tuple[float, float]:
    """
    Calcs the True Positive Rate (TPR) and False Positive Rate (FPR) 
    for a given threshold, where Spam (1) is the positive class.
    
    Args:
        df: DataFrame containing 'Spam/Ham' (True Labels) and 'spam' (Probabilities).
        threshold: the classification cutoff (predicted spam probability >= threshold --> predict Spam).
        
    Returns:
         tuple (FPR, TPR).
    """
    
    #  predictions based on  threshold
    predictions = (df['spam'] >= threshold).astype(int)
    
    # True labels
    y_true      = df['Spam/Ham']
    
    #  Confusion Matrix: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels = [0, 1]).ravel()
    
    # Calculate True Positive Rate (TPR or Recall)
    total_positives = tp + fn
    tpr             = tp / total_positives if total_positives > 0 else 0.0 # division by zero
    
    # False Positive Rate (FPR)
    total_negatives = tn + fp
    fpr             = fp / total_negatives if total_negatives > 0 else 0.0
    
    return fpr, tpr


## Part 1: Download

In [130]:
# !wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

In [131]:
# pandas can read the zip file directly
path            = 'https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip'
enron_spam_data = pd.read_csv(path, compression='zip')

# enron_spam_data = pd.read_csv("/Users/arunram/Desktop/Data Science/BU_MSDS/Module 8/dx704-project-09-main/enron_spam_data.csv")
enron_spam_data.head(10)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,1999-12-14
6,6,meter 1517 - jan 1999,"george ,\ni need the following done :\njan 13\...",ham,1999-12-14
7,7,duns number changes,fyi\n- - - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
8,8,king ranch,there are two fields of gas that i am having d...,ham,1999-12-14
9,9,re : entex transistion,thanks so much for the memo . i would like to ...,ham,1999-12-14


In [132]:
# print((enron_spam_data["Spam/Ham"] == "spam").mean())
enron_spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Message ID  33716 non-null  int64 
 1   Subject     33427 non-null  object
 2   Message     33345 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [133]:
df = enron_spam_data.copy()
df.groupby('Spam/Ham').describe()


Message ID                                                       \
              count          mean          std     min      25%      50%   
Spam/Ham                                                                   
ham         16545.0  11765.862315  8279.563017     0.0   5636.0  11268.0   
spam        17171.0  21763.512783  8421.126803  3672.0  16337.5  22130.0   

                            
              75%      max  
Spam/Ham                    
ham       16904.0  29215.0  
spam      29422.5  33715.0

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the `Subject` and `Message` columns are relevant sources of text data.

- For each email, you should count the number of repetitions of each feature present.

- The auto-grader will assume that you are using a *multinomial distribution* in the following problems.

- Assign a row to the TEST data set if `Message ID % 30 == 0` and assign it to the TRAINING data set otherwise.

- Write two files, "`train-features.tsv`" and "`test-features.tsv`" with two columns, `Message ID` and `features_json`.

- The `features_json` column should contain a *JSON dictionary* where the keys are the feature names and the values are integers representing the number of occurrences of that feature.
    - <font color = 'cyan'> Why in json ? Because it guarantees that the column will be treated in the Dataframe as a string, a simpler datatype than a python dictionary. Easy for storage, easy for for lightweight data interchange (ie. to a .tsv file), easy for the Autograder to read. Python dictionaries are fine to use in a Dataframe in-app, but not if the Dataframe needs to be interchanged. </font>
- This will give us a sparse feature representation.

<p style="color: red;">Red Text</p>

In [134]:
df['Spam/Ham'] = df['Spam/Ham'].map({'spam': 1, 'ham': 0})
df.head(5)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,0,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",0,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,0,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,1999-12-14


In [135]:

print("Combining Subject and Message, and extracting features...")

# new column combining two text fields with a space in between
df['Full_Text']     = df['Subject'].fillna('') + ' ' + df['Message'].fillna('')

# Apply feature extraction function to new combined text column, then convert each dict in df['features_dict] to a JSON string
df['features_dict'] = df['Full_Text'].apply(process_and_extract_features)
df['features_json'] = df['features_dict'].apply(json.dumps)

print(f'Number of rows after feature extraction: {len(df)}')
print("\nthis is so f*cking complicated...")

Combining Subject and Message, and extracting features...
Extracting 4 features from text of length 29
Extracting 147 features from text of length 4307
Extracting 5 features from text of length 67
Extracting 59 features from text of length 1182
Extracting 66 features from text of length 1150
Extracting 49 features from text of length 559
Extracting 39 features from text of length 433
Extracting 107 features from text of length 1395
Extracting 97 features from text of length 1627
Extracting 205 features from text of length 3062
Extracting 123 features from text of length 1752
Extracting 94 features from text of length 1067
Extracting 64 features from text of length 738
Extracting 54 features from text of length 619
Extracting 31 features from text of length 311
Extracting 62 features from text of length 853
Extracting 63 features from text of length 511
Extracting 56 features from text of length 651
Extracting 61 features from text of length 1018
Extracting 34 features from text of leng

In [136]:
df.head(10)

,Message ID,Subject,Message,Spam/Ham,Date,Full_Text,features_dict,features_json
0,0,christmas tree farm pictures,NaN,0,1999-12-10,christmas tree farm pictures,"{'christmas': 1, 'tree': 1, 'farm': 1, 'pictur...","{""christmas"": 1, ""tree"": 1, ""farm"": 1, ""pictur..."
1,1,"vastar resources , inc .","gary , production from the high island larger ...",0,1999-12-13,"vastar resources , inc . gary , production fro...","{'vastar': 6, 'resources': 4, 'inc': 4, 'gary'...","{""vastar"": 6, ""resources"": 4, ""inc"": 4, ""gary""..."
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,1999-12-14,calpine daily gas nomination - calpine daily g...,"{'calpine': 2, 'daily': 2, 'gas': 2, 'nominati...","{""calpine"": 2, ""daily"": 2, ""gas"": 2, ""nominati..."
3,3,re : issue,fyi - see note below - already done .\nstella\...,0,1999-12-14,re : issue fyi - see note below - already done...,"{'re': 2, 'issue': 4, 'fyi': 1, 'see': 1, 'not...","{""re"": 2, ""issue"": 4, ""fyi"": 1, ""see"": 1, ""not..."
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,1999-12-14,meter 7268 nov allocation fyi .\n- - - - - - -...,"{'meter': 3, '7268': 3, 'nov': 3, 'allocation'...","{""meter"": 3, ""7268"": 3, ""nov"": 3, ""allocation""..."
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",0,1999-12-14,"mcmullen gas for 11 / 99 jackie ,\nsince the i...","{'mcmullen': 2, 'gas': 5, '11': 1, '99': 2, 'j...","{""mcmullen"": 2, ""gas"": 5, ""11"": 1, ""99"": 2, ""j..."
6,6,meter 1517 - jan 1999,"george ,\ni need the following done :\njan 13\...",0,1999-12-14,"meter 1517 - jan 1999 george ,\ni need the fol...","{'meter': 1, '1517': 1, 'jan': 3, '1999': 1, '...","{""meter"": 1, ""1517"": 1, ""jan"": 3, ""1999"": 1, ""..."
7,7,duns number changes,fyi\n- - - - - - - - - - - - - - - - - - - - -...,0,1999-12-14,duns number changes fyi\n- - - - - - - - - - -...,"{'duns': 2, 'number': 5, 'changes': 3, 'fyi': ...","{""duns"": 2, ""number"": 5, ""changes"": 3, ""fyi"": ..."
8,8,king ranch,there are two fields of gas that i am having d...,0,1999-12-14,king ranch there are two fields of gas that i ...,"{'king': 5, 'ranch': 6, 'there': 4, 'two': 1, ...","{""king"": 5, ""ranch"": 6, ""there"": 4, ""two"": 1, ..."
9,9,re : entex transistion,thanks so much for the memo . i would like to ...,0,1999-12-14,re : entex transistion thanks so much for the ...,"{'re': 1, 'entex': 8, 'transistion': 3, 'thank...","{""re"": 1, ""entex"": 8, ""transistion"": 3, ""thank..."


In [137]:
# TRAIN AND TEST SPLIT rule: 'Message ID' % 30 == 0 is TEST, otherwise TRAIN
print("Splitting data into TRAIN and TEST sets...")

split_rule_train    = df['Message ID'] % 30 != 0
df_train            = df[split_rule_train].copy()

split_rule_test     = df['Message ID'] % 30 == 0
df_test             = df[split_rule_test].copy()

print(f"Training row count: {len(df_train)}")
print(f"Testing row count: {len(df_test)}")

# df_test.to_csv('test-df.csv', index=False, header=True)

# df_test

Splitting data into TRAIN and TEST sets...
Training row count: 32592
Testing row count: 1124


In [138]:
df_train.head()

,Message ID,Subject,Message,Spam/Ham,Date,Full_Text,features_dict,features_json
1,1,"vastar resources , inc .","gary , production from the high island larger ...",0,1999-12-13,"vastar resources , inc . gary , production fro...","{'vastar': 6, 'resources': 4, 'inc': 4, 'gary'...","{""vastar"": 6, ""resources"": 4, ""inc"": 4, ""gary""..."
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,1999-12-14,calpine daily gas nomination - calpine daily g...,"{'calpine': 2, 'daily': 2, 'gas': 2, 'nominati...","{""calpine"": 2, ""daily"": 2, ""gas"": 2, ""nominati..."
3,3,re : issue,fyi - see note below - already done .\nstella\...,0,1999-12-14,re : issue fyi - see note below - already done...,"{'re': 2, 'issue': 4, 'fyi': 1, 'see': 1, 'not...","{""re"": 2, ""issue"": 4, ""fyi"": 1, ""see"": 1, ""not..."
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,1999-12-14,meter 7268 nov allocation fyi .\n- - - - - - -...,"{'meter': 3, '7268': 3, 'nov': 3, 'allocation'...","{""meter"": 3, ""7268"": 3, ""nov"": 3, ""allocation""..."
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",0,1999-12-14,"mcmullen gas for 11 / 99 jackie ,\nsince the i...","{'mcmullen': 2, 'gas': 5, '11': 1, '99': 2, 'j...","{""mcmullen"": 2, ""gas"": 5, ""11"": 1, ""99"": 2, ""j..."


In [139]:
# YOUR CHANGES HERE

# save cols from Training Data
train_features_df = df_train[['Message ID', 'features_json']].copy()
train_features_df.to_csv('train-features.tsv', sep = '\t', index = False)
print("\nWrote 'train-features.tsv'")

# train_features_df = prepare_output_dataframe(df_train)

# save cols from Testing Data
test_features_df = df_test[['Message ID', 'features_json']].copy()
test_features_df.to_csv('test-features.tsv', sep = '\t', index = False)
print("Wrote 'test-features.tsv'")


Wrote 'train-features.tsv'
Wrote 'test-features.tsv'


Submit "`train-features.tsv`" and "`test-features.tsv`" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your **TRAINING** data, compute appropriate conditional probabilities for use with naïve Bayes.
Use *additive smoothing* with $\alpha=1$ to avoid zeros.


Save the conditional probabilities in a file "`feature-probabilities.tsv`" with columns `feature`, `ham_probability` and `spam_probability`.

In [140]:
# df['Label'] = df['Spam/Ham'].map({'spam': 1, 'ham': 0})

In [141]:
# Set smoothing parameter (alpha=1 for Laplace smoothing)
ALPHA = 1

# Train Naive Bayes model and get likelihood DataFrame
likelihood_df = train_naive_bayes_model(df_train, feature_col = 'features_dict', label_col = 'Spam/Ham', alpha = ALPHA)


Training Naive Bayes Model and generating df of feature probabilities...


In [142]:
likelihood_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155840 entries, 0 to 155839
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   feature           155840 non-null  object 
 1   ham_probability   155840 non-null  float64
 2   spam_probability  155840 non-null  float64
dtypes: float64(2), object(1)
memory usage: 3.6+ MB


In [143]:
likelihood_df.to_csv('feature-probabilities.tsv', sep = '\t', index = False)

Submit "`feature-probabilities.tsv`" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a <font color = 'cyan'>naïve Bayes classifier</font> based on your previous feature probabilities. Save your prediction probabilities to "`train-predictions.tsv`" with columns `Message ID`, `ham` and `spam`.

In [144]:
total_emails_train = len(df_train)
total_emails_test  = len(df_test)

ham_count_train  = (df_train['Spam/Ham'] == 0).sum()
spam_count_train = (df_train['Spam/Ham'] == 1).sum()
print(f"Total training emails: {total_emails_train}. | (Ham: {ham_count_train}, Spam: {spam_count_train})")

# Priors (P(Spam) and P(Ham))
PRIOR_HAM  = ham_count_train / total_emails_train
PRIOR_SPAM = spam_count_train / total_emails_train

print(f"Prior P(Ham): {PRIOR_HAM:.4f}")
print(f"Prior P(Spam): {PRIOR_SPAM:.4f}")

PRIORS = {0: PRIOR_HAM, 1: PRIOR_SPAM}

# ____________________________________ #


prior_ham_ = (enron_spam_data["Spam/Ham"] == "ham").mean()
print(f'Prior P(Ham) check: {prior_ham_:.4f}')

prior_spam_ = (enron_spam_data["Spam/Ham"] == "spam").mean()
print(f'Prior P(Spam) check: {prior_spam_:.4f}')
# LOG_PRIOR_HAM   = np.log(PRIOR_HAM)
# LOG_PRIOR_SPAM  = np.log(PRIOR_SPAM)
# print(f'Log Prior P(Spam): {LOG_PRIOR_SPAM:.4f}')
# print(f'Log Prior P(Ham): {LOG_PRIOR_HAM:.4f}')


Total training emails: 32592. | (Ham: 15993, Spam: 16599)
Prior P(Ham): 0.4907
Prior P(Spam): 0.5093
Prior P(Ham) check: 0.4907
Prior P(Spam) check: 0.5093


In [145]:
# Convert likelihood_df to a model lookup structure for fast access:
# model_likelihoods maps: feature -> {0: ham_prob, 1: spam_prob}
# This structure is needed to look up P(f|C) during classification.

# a goddamn hornet's nest of nested dictionaries coupled with f*cking lambda statements which I despise more than yellowdig....
# model_likelihoods = likelihood_df.set_index('feature').apply(lambda row: {
#     0: row['ham_probability'], 
#     1: row['spam_probability']
# }, axis = 1).to_dict()

model_likelihoods = {}

for _, row in likelihood_df.iterrows():
    feature = row['feature']
    ham_prob = row['ham_probability']
    spam_prob = row['spam_probability']
    model_likelihoods[feature] = {0: ham_prob, 1: spam_prob}

MODEL = {
    'prior':      PRIORS,
    'likelihood': model_likelihoods
}

In [146]:
# # 1. Create a temporary DataFrame with columns renamed to the desired inner dict keys (0 and 1)
# temp_df = likelihood_df.rename(columns={
#     'ham_probability': 0,
#     'spam_probability': 1
# })

# # 2. Set the 'feature' column as the index, and convert to a dictionary of dictionaries
# model_likelihoods = temp_df.set_index('feature')[[0, 1]].to_dict('index')

# # Your MODEL dictionary remains the same
# MODEL = {
#     'prior': PRIORS,
#     'likelihood': model_likelihoods
# }

In [147]:
# Run Classification on  TRAINING set (df_train)
results          = df_train['features_dict'].apply(lambda x: classify_document(x, MODEL))
# results

In [148]:
df['ham'] = results.str[0]
df['spam'] = results.str[1]

In [149]:
# df[['x', 'y']] = pd.DataFrame(df['pairs'].tolist(), index=df.index)

df_train[['ham', 'spam']] = pd.DataFrame(results.tolist(), index = df_train.index)
# Separate results into two columns (alternate way)
# df_train['ham']      = results.apply(lambda x: x[0])
# df_train['spam']     = results.apply(lambda x: x[1])

df_train.head()

,Message ID,Subject,Message,Spam/Ham,Date,Full_Text,features_dict,features_json,ham,spam
1,1,"vastar resources , inc .","gary , production from the high island larger ...",0,1999-12-13,"vastar resources , inc . gary , production fro...","{'vastar': 6, 'resources': 4, 'inc': 4, 'gary'...","{""vastar"": 6, ""resources"": 4, ""inc"": 4, ""gary""...",1.0,2.972802e-177
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,1999-12-14,calpine daily gas nomination - calpine daily g...,"{'calpine': 2, 'daily': 2, 'gas': 2, 'nominati...","{""calpine"": 2, ""daily"": 2, ""gas"": 2, ""nominati...",1.0,1.498243e-12
3,3,re : issue,fyi - see note below - already done .\nstella\...,0,1999-12-14,re : issue fyi - see note below - already done...,"{'re': 2, 'issue': 4, 'fyi': 1, 'see': 1, 'not...","{""re"": 2, ""issue"": 4, ""fyi"": 1, ""see"": 1, ""not...",1.0,2.599343e-155
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,1999-12-14,meter 7268 nov allocation fyi .\n- - - - - - -...,"{'meter': 3, '7268': 3, 'nov': 3, 'allocation'...","{""meter"": 3, ""7268"": 3, ""nov"": 3, ""allocation""...",1.0,1.276878e-148
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",0,1999-12-14,"mcmullen gas for 11 / 99 jackie ,\nsince the i...","{'mcmullen': 2, 'gas': 5, '11': 1, '99': 2, 'j...","{""mcmullen"": 2, ""gas"": 5, ""11"": 1, ""99"": 2, ""j...",1.0,1.466954e-38


In [150]:
predictions_train_df = df_train[['Message ID', 'ham', 'spam']].copy()
predictions_train_df.head()

,Message ID,ham,spam
1,1,1.0,2.972802e-177
2,2,1.0,1.498243e-12
3,3,1.0,2.599343e-155
4,4,1.0,1.276878e-148
5,5,1.0,1.466954e-38


In [151]:

file_name  = "train-predictions.tsv"
predictions_train_df.to_csv(file_name, sep = '\t', index = False)

print(f"Prediction probabilities saved to '{file_name}'.")

Prediction probabilities saved to 'train-predictions.tsv'.


Submit "`train-predictions.tsv`" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the *TEST* data.
Save your prediction probabilities in "`test-predictions.tsv`" with the same columns as "train-predictions.tsv".

In [152]:

results_test          = df_test['features_dict'].apply(lambda x: classify_document(x, MODEL))
# results_test.head()

In [153]:
df_test['ham']  = results_test.str[0]
df_test['spam'] = results_test.str[1]
df_test.head()

,Message ID,Subject,Message,Spam/Ham,Date,Full_Text,features_dict,features_json,ham,spam
0,0,christmas tree farm pictures,NaN,0,1999-12-10,christmas tree farm pictures,"{'christmas': 1, 'tree': 1, 'farm': 1, 'pictur...","{""christmas"": 1, ""tree"": 1, ""farm"": 1, ""pictur...",0.056785,9.432148e-01
30,30,january production estimate,daren / carlos :\ni did not receive any more c...,0,1999-12-20,january production estimate daren / carlos :\n...,"{'january': 3, 'production': 3, 'estimate': 5,...","{""january"": 3, ""production"": 3, ""estimate"": 5,...",1.000000,8.333413e-83
60,60,new pooling point # 7342,the new pooling point will be handled by rober...,0,1999-12-28,new pooling point # 7342 the new pooling point...,"{'new': 2, 'pooling': 2, 'point': 2, '7342': 1...","{""new"": 2, ""pooling"": 2, ""point"": 2, ""7342"": 1...",1.000000,1.950794e-12
90,90,"enron actuals for january 4 , 2000",teco tap 110 . 000 / hpl iferc\nteco tap 33 . ...,0,2000-01-05,"enron actuals for january 4 , 2000 teco tap 11...","{'enron': 3, 'actuals': 1, 'january': 1, '2000...","{""enron"": 3, ""actuals"": 1, ""january"": 1, ""2000...",1.000000,6.309313e-34
120,120,re : revision - black marlin - meter 986782 fo...,"yes should be , right howard ?\nstella\n- - - ...",0,2000-01-11,re : revision - black marlin - meter 986782 fo...,"{'re': 2, 'revision': 5, 'black': 4, 'marlin':...","{""re"": 2, ""revision"": 5, ""black"": 4, ""marlin"":...",1.000000,6.397067e-186


In [154]:
prediction_df_test = df_test[['Message ID', 'ham', 'spam']].copy()


file_name = "test-predictions.tsv"
prediction_df_test.to_csv(file_name, sep='\t', index=False)

Submit "`test-predictions.tsv`" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the TEST data using the `spam` class for positives.
That is, if the <font color = 'cyan'> predicted spam probability is >= to the threshold, predict spam. </font>

Save this data in a file "`roc.tsv`" with columns `threshold`, `false_positive_rate` and `true_positive rate`.

In [155]:

#  thresholds  with 0.01 increments
thresholds  = np.arange(0.01, 1.00, 0.01)

roc_data    = []

print("Calculating FPR and TPR across 99 thresholds...")

for threshold in thresholds:
    fpr, tpr = calculate_roc_curve(df_test, threshold)
    
    roc_data.append({
        'threshold': round(threshold, 2),
        'false_positive_rate': fpr,
        'true_positive_rate': tpr
    })

roc_df = pd.DataFrame(roc_data)

print(f"ROC data calculated for {len(roc_df)} points.")

roc_df

Calculating FPR and TPR across 99 thresholds...
ROC data calculated for 99 points.


,threshold,false_positive_rate,true_positive_rate
0,0.01,0.034420,0.998252
1,0.02,0.032609,0.998252
2,0.03,0.027174,0.998252
3,0.04,0.025362,0.998252
4,0.05,0.025362,0.998252
...,...,...,...
94,0.95,0.010870,0.972028
95,0.96,0.010870,0.970280
96,0.97,0.010870,0.966783
97,0.98,0.010870,0.965035


In [156]:
file_name = "roc.tsv"
roc_df.to_csv(file_name, sep = '\t', index = False)

Submit "`roc.tsv`" in Gradescope.

## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

## Misc. code

In [157]:
#  Run Classification on df_test ---
# print("Classifying Test Documents...")
# # Use the correct feature column name: 'features_dict'
# df_test['P(Spam|Document)'] = df_test['features_dict'].apply(lambda x: classify_document(x, MODEL))

# Evaluate Performance (Default Threshold = 0.5) ---
# THRESHOLD = 0.5
# df_test['Prediction'] = (df_test['P(Spam|Document)'] > THRESHOLD).astype(int)

# Use the correct label column name: 'Spam/Ham'
# y_true = df_test['Spam/Ham'] 
# y_pred = df_test['Prediction']


In [158]:

# print(f"\n--- Model Performance on Test Set (Threshold = {THRESHOLD}) ---")
# print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
# print(f"Precision (Spam): {precision_score(y_true, y_pred, zero_division=0):.4f}")
# print(f"Recall (Spam): {recall_score(y_true, y_pred, zero_division=0):.4f}")
# print(f"F1 Score (Spam): {f1_score(y_true, y_pred, zero_division=0):.4f}")

# Exploring Threshold Tradeoffs ---
# thresholds = np.linspace(0.01, 0.99, 10)
# results_list = []

# for t in thresholds:
#     y_pred_t = (df_test['P(Spam|Document)'] >= t).astype(int)
    
#     if sum(y_pred_t) == 0:
#         p = 1.0 
#         r = 0.0
#         f = 0.0
#         acc = accuracy_score(y_true, y_pred_t)
#     else:
#         p = precision_score(y_true, y_pred_t, zero_division=0)
#         r = recall_score(y_true, y_pred_t, zero_division=0)
#         f = f1_score(y_true, y_pred_t, zero_division=0)
#         acc = accuracy_score(y_true, y_pred_t)

#     results_list.append({
#         'Threshold': t,
#         'Accuracy': acc,
#         'Precision': p,
#         'Recall': r,
#         'F1 Score': f
#     })

# results_df = pd.DataFrame(results_list)
# print("\n--- Model Performance Across Different Thresholds ---")
# results_df.round(4)

In [159]:

# Priors (P(Spam) and P(Ham)) ---
# #   needed for the log-likelihood calculation: log(P(Class)) + sum(log(P(Word|Class)))
# target_df = df_train.copy()
# # Calculate  log priors
# LOG_PRIOR_SPAM  = np.log(P_SPAM)
# LOG_PRIOR_HAM   = np.log(P_HAM)
 
# print(f"Log Prior P(Spam): {LOG_PRIOR_SPAM:.4f}")
# print(f"Log Prior P(Ham): {LOG_PRIOR_HAM:.4f}")

In [160]:
# Load Conditional Probabilities (P(Word|Class)) ---


# # Pre-calculate the log probabilities and store them in a dictionary for fast lookup
# # {feature: {ham_prob: log(P(Word|Ham)), spam_prob: log(P(Word|Spam))}}


# # 1. Convert DataFrame to a Dictionary of Dictionaries
# # We use 'feature' as the key to look up probabilities quickly.
# feature_probabilities_df['log_ham_prob']    = np.log(feature_probabilities_df['ham_probability'])
# feature_probabilities_df['log_spam_prob']   = np.log(feature_probabilities_df['spam_probability'])

# prob_map = feature_probabilities_df.set_index('feature')[['log_ham_prob', 'log_spam_prob']].to_dict('index')

# print(f"Loaded {len(prob_map):,} feature probabilities.")


In [161]:
# # --- 2. Classification Function ---

# def classify_email(features_dict, prob_map, log_prior_spam, log_prior_ham):
#     """
#     Calculates the log-likelihood scores for an email being spam and ham.
    
#     Args:
#         features_dict (dict): Dictionary of word counts for the email.
#         prob_map (dict): Dictionary mapping feature names to their log probabilities.
#         log_prior_spam (float): Log of the prior probability P(Spam).
#         log_prior_ham (float): Log of the prior probability P(Ham).
        
#     Returns:
#         tuple: (log_spam_score, log_ham_score)
#     """
    
#     # Initialize scores with the log priors
#     log_spam_score = log_prior_spam
#     log_ham_score = log_prior_ham
    
#     # Calculate the sum of log conditional probabilities
#     for feature, count in features_dict.items():
        
#         if feature in prob_map:
#             log_probs = prob_map[feature]
            
#             # Add the feature's log probability multiplied by its count
#             log_spam_score += count * log_probs['log_spam_prob']
#             log_ham_score += count * log_probs['log_ham_prob']
            
#     return log_spam_score, log_ham_score


In [162]:
# # --- 3. Apply Classification to Training Set ---

# print("Classifying training data and computing log scores...")

# # Apply the classification function to the 'features_dict' column of the training set
# scores = target_df['features_dict'].apply(
#     lambda x: classify_email(x, prob_map, LOG_PRIOR_SPAM, LOG_PRIOR_HAM)
# )

# # Split the resulting tuple into two columns for log-scores
# target_df['log_spam_score'] = scores.apply(lambda x: x[0])
# target_df['log_ham_score'] = scores.apply(lambda x: x[1])

In [163]:
# # ---  Convert Log Scores to Final Probabilities (P(Class|Email)) ---

# # Calculate the denominator for normalization: log(P(Spam) + P(Ham))
# target_df['log_sum']            = np.logaddexp(target_df['log_spam_score'], target_df['log_ham_score'])

# # Calculate final probabilities: P(Spam | Email) = exp(log_spam_score - log_sum)
# target_df['spam_probability']   = np.exp(target_df['log_spam_score'] - target_df['log_sum'])
# target_df['ham_probability']    = np.exp(target_df['log_ham_score'] - target_df['log_sum'])

# target_df.head()

In [164]:
# # YOUR CHANGES HERE

# # Prepare the final output DataFrame with the required columns
# output_df           = target_df[['Message ID', 'ham_probability', 'spam_probability']].copy()
# output_df.columns   = ['Message ID', 'ham', 'spam'] # Rename columns as required: 'ham' and 'spam'

# # Save the DataFrame to the REQUIRED Part 4 output file: train-predictions.tsv
# output_df.to_csv('train-predictions.tsv', sep='\t', index=False)